In [1]:
import os
os.chdir("/workspaces/pythonProject/medallion-turso-v3/notebooks")

# 04 · Multi-source pipeline: five sources → one customer
```
Turso source tables      Bronze (per source)          Silver (per source)            Silver (combined)            Gold
crm_accounts          →  bronze_crm_accounts       →  silver_crm_accounts          ─┐
billing_customers     →  bronze_billing_customers  →  silver_billing_customers     ─┤
support_contacts      →  ...                       →  ...                           ├→ silver_customer_unified → gold_dim_customer_360
web_visitors          →  ...                       →  ...                           │   (match on email,
marketing_leads       →  ...                       →  ...                          ─┘    precedence per attribute)
```
Bronze and Silver keep each source separate, so one bad feed never blocks the others. The combine step matches people **by normalised email** and picks each attribute from the highest-priority source that has it (the `from:` lists in the YAML). Reuses every helper from `00_config` (stage, clean, merge, watermarks); this notebook only adds the combine logic.

The five sources, the unified customer (which source wins for each attribute) and the Gold dimension, with all their columns and types, are defined in **`domains/customer360.yml`**; this notebook contains only the logic.

In [2]:
%run -i ./00_config.ipynb

rule library: 27 checks for 9 entities


## Load the customer360 domain
`load_domain("customer360")` registers the five sources (`ENTITIES`), the unified customer (`UNIFIED`) and `dim_customer_360` (`DIMS`) from `domains/customer360.yml`. Data-quality rules for the five sources are in `rules/dq_rules.yml`; rows without an email cannot be matched, so a not-null rule on `email` sends them to `silver_quarantine`.

In [3]:
C360 = load_domain("customer360")
SOURCES = C360["entities"]          # the five source entities

domain 'customer360': 5 entities, 1 dimensions, 0 facts


In [4]:
open_warehouse()
new_batch()
print(f"run batch {BATCH_ID} | warehouse: Turso")

run batch 4 | warehouse: Turso


## 1 · Stage and 2 · Clean, one source at a time

In [5]:
for e in SOURCES: stage(e)

  crm_accounts       extracted     1 rows (since 2026-10-30 14:13:55),     0 new in bronze
  billing_customers  extracted     1 rows (since 2026-10-31 20:29:53),     0 new in bronze
  support_contacts   extracted     1 rows (since 2026-10-31 21:16:00),     0 new in bronze
  web_visitors       extracted     1 rows (since 2026-10-31 18:55:13),     0 new in bronze
  marketing_leads    extracted     1 rows (since 2026-10-28 20:20:37),     0 new in bronze


In [6]:
for e in SOURCES: clean_to_silver(e)

  crm_accounts       nothing new in bronze
  billing_customers  nothing new in bronze
  support_contacts   nothing new in bronze
  web_visitors       nothing new in bronze
  marketing_leads    nothing new in bronze


## 3 · Combine: one row per person
`affected` = emails whose row changed in any source since the last run (every email on the first run or in `full` mode). For those emails, take the latest Silver row from each source, join them on email, and build each attribute with `coalesce` in the order listed in the YAML (first source that has a value wins). The result is merged into `silver_customer_unified`; only people whose values changed are rewritten.

> Limitation: an email *change* in a source is an identity change. This example treats the new email as a new person; handling it needs an identity-resolution step (a stable person id).

In [7]:
def build_unified(name):
    """Combine the sources into one row per person, driven entirely by UNIFIED[name] (from the YAML)."""
    u = UNIFIED[name]; key, m, srcs = u["key"], u["match_on"], u["sources"]          # srcs: alias -> (entity, entity key)
    wm = get_wm(f"silver.{name}")
    if LOAD_MODE == "full" or wm is None:
        affected = " UNION ".join(f"SELECT {m} FROM silver_{t}" for t, _ in srcs.values())
    else:
        affected = " UNION ".join(f"SELECT {m} FROM silver_{t} WHERE _silver_batch > {int(wm)}" for t, _ in srcs.values())
    latest = ",\n".join(f"""{a} AS (SELECT * FROM (SELECT t.*, row_number() OVER (PARTITION BY {m} ORDER BY updated_at DESC, {k} DESC) AS _rn
                 FROM silver_{t} t WHERE {m} IN (SELECT {m} FROM affected)) WHERE _rn = 1)"""
                        for a, (t, k) in srcs.items())
    sel = {key: f"e.{m}", m: f"e.{m}"}                                                  # the person key is the normalised email
    for col, _, frm in u["attributes"]: sel[col] = ("coalesce(" + ", ".join(frm) + ")") if len(frm) > 1 else frm[0]      # first non-null source wins
    for flag, a in u["flags"]:          sel[flag] = f"({a}.{m} IS NOT NULL)"
    sel["source_count"] = " + ".join(f"({a}.{m} IS NOT NULL)" for a in srcs)
    joins = " ".join(f"LEFT JOIN {a} ON {a}.{m} = e.{m}" for a in srcs)
    cols = [c for c, _ in u["columns"]]
    _work("wrk_unified", f"WITH affected AS ({affected}), {latest} SELECT {', '.join(f'{sel[c]} AS {c}' for c in cols)} FROM affected e {joins}")
    con.execute(f"""CREATE TABLE IF NOT EXISTS silver_{name}
        ({', '.join(f'{n} {sqltype(t)}' for n, t in u['columns'])}, _silver_batch INTEGER, PRIMARY KEY ({key}))""")
    n = con.scalar("SELECT count(*) FROM wrk_unified")
    changed = merge(f"silver_{name}", "wrk_unified", [key], cols, "_silver_batch")
    set_wm(f"silver.{name}", BATCH_ID); con.commit()
    print(f"  {name:17s} {n:5d} people re-evaluated -> {changed:5d} inserted/updated")

for u in C360["unified"]: build_unified(u)

  customer_unified      0 people re-evaluated ->     0 inserted/updated


## 4 · Merge into Gold: `gold_dim_customer_360`
The same generic `load_dim` used for the sales dimensions: only people changed in `silver_customer_unified` since the last Gold load are merged. The `lifecycle` column is a *derived attribute* defined in the YAML (`expr`), and `crm_account_id` is the bridge back to the sales data.

In [8]:
for d in C360["dims"]: load_dim(d)

con.execute("DROP VIEW IF EXISTS gold_v_customer_source_overlap")
con.execute("""CREATE VIEW gold_v_customer_source_overlap AS
    SELECT trim(CASE WHEN in_crm THEN '+crm' ELSE '' END || CASE WHEN in_billing THEN '+billing' ELSE '' END ||
                CASE WHEN in_support THEN '+support' ELSE '' END || CASE WHEN in_web THEN '+web' ELSE '' END ||
                CASE WHEN in_marketing THEN '+marketing' ELSE '' END, '+') AS sources, count(*) AS customers
    FROM gold_dim_customer_360 WHERE customer_sk <> -1 GROUP BY 1 ORDER BY 2 DESC""")
con.commit()

  dim_customer_360     0 inserted/updated


## 5 · Checks

In [9]:
display(con.df("SELECT * FROM ctl_watermark WHERE name LIKE '%customer%' OR name LIKE '%crm%' OR name LIKE '%billing%' ORDER BY name"))
display(con.df("SELECT entity, count(*) AS quarantined FROM silver_quarantine WHERE entity IN (" + ",".join(f"'{s}'" for s in SOURCES) + ") GROUP BY 1 ORDER BY 1"))
display(con.df("SELECT lifecycle, count(*) AS customers FROM gold_dim_customer_360 WHERE customer_sk <> -1 GROUP BY 1 ORDER BY 2 DESC"))
display(con.df("SELECT * FROM gold_v_customer_source_overlap"))

,name,value,updated_ts
0,extract.billing_customers,2026-10-31 20:29:53,2026-11-19 03:13:23
1,extract.crm_accounts,2026-10-30 14:13:55,2026-11-19 03:13:22
2,extract.customers,2026-10-30 14:13:55,2026-11-19 03:11:44
3,gold.dim_customer,1,2026-11-19 03:11:46
4,gold.dim_customer_360,2,2026-11-19 03:13:24
5,silver.billing_customers,2,2026-11-19 02:43:48
6,silver.crm_accounts,2,2026-11-19 02:43:47
7,silver.customer_unified,4,2026-11-19 03:13:24
8,silver.customers,1,2026-11-19 02:41:03


,entity,quarantined
0,billing_customers,1
1,crm_accounts,4
2,web_visitors,4


,lifecycle,customers
0,customer,38
1,lead,25
2,prospect,18
3,visitor or contact,10
4,former customer,5


,sources,customers
0,marketing,25
1,crm+billing+support,12
2,crm+billing,10
3,web,6
4,crm+billing+support+web,6
5,crm,5
6,billing,5
7,support,4
8,crm+billing+web,4
9,crm+web,3


In [10]:
# Where sources disagree, the precedence rules decide: CRM name vs billing name for the same email
display(con.df("""
  SELECT c.email, c.full_name AS crm_name, b.billing_name, d.full_name AS gold_name
  FROM silver_crm_accounts c JOIN silver_billing_customers b USING (email)
  JOIN gold_dim_customer_360 d ON d.email = c.email
  WHERE lower(c.full_name) <> lower(b.billing_name) LIMIT 8"""))
display(con.df("SELECT * FROM gold_dim_customer_360 WHERE source_count = 5 LIMIT 3"))

,email,crm_name,billing_name,gold_name
0,pavel.cohen25@example.com,Pavel Cohen,P. Cohen,Pavel Cohen
1,vik.jones50@example.com,Vik Jones,V. Jones,Vik Jones
2,frank.meyer54@example.com,Frank Meyer,F. Meyer,Frank Meyer
3,wen.novak55@example.com,Wen Novak,W. Novak,Wen Novak


,customer_sk,customer_key,email,full_name,phone,segment,country,lifecycle,plan,billing_status,...,lead_status,marketing_consent,crm_account_id,in_crm,in_billing,in_support,in_web,in_marketing,source_count,_gold_batch
0,66,uma.garcia20@example.com,uma.garcia20@example.com,Uma Garcia,6172919159,retail,GB,customer,pro,active,...,new,N,20,1,1,1,1,1,5,2


## DQ report for the five sources (rules from `dq_rules.yml`)

In [11]:
display(con.df("""SELECT entity, rule, criticality, sum(failed_rows) AS failed_rows, sum(checked_rows) AS checked_rows
                  FROM silver_dq_results WHERE entity IN (""" + ",".join(f"'{s}'" for s in SOURCES) + """)
                  GROUP BY entity, rule, criticality HAVING sum(failed_rows) > 0 ORDER BY entity, rule"""))
# optional: same YAML on native DQX / Spark (skipped automatically if Java or Spark is unavailable)
for e in SOURCES: native_dqx_check(e)

,entity,rule,criticality,failed_rows,checked_rows
0,billing_customers,mrr_non_negative,error,1,47
1,crm_accounts,email_is_not_null,error,4,60
2,web_visitors,email_is_not_null,error,3,31
3,web_visitors,visits_non_negative,error,1,31


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/19 03:13:27 WARN Utils: Your hostname, codespaces-f42fe3, resolves to a loopback address: 127.0.0.1; using 10.0.2.156 instead (on interface eth0)
26/11/19 03:13:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 03:13:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


  crm_accounts       SQL errors/warnings = 4/0 | native DQX = 4/0 -> MATCH
  billing_customers  SQL errors/warnings = 1/0 | native DQX = 1/0 -> MATCH
  support_contacts   SQL errors/warnings = 0/0 | native DQX = 0/0 -> MATCH
  web_visitors       SQL errors/warnings = 4/0 | native DQX = 4/0 -> MATCH
  marketing_leads    SQL errors/warnings = 0/0 | native DQX = 0/0 -> MATCH


In [12]:
release_all()          # closes the Turso connection (the next notebook opens its own)